# Overlay clock legibility at 640x360

The procedure plan wants to feed the model `overlayed/` video so it can read an
absolute `HH:MM:SS` straight off each frame instead of associating a text marker with
a frame hundreds of tokens away.

That only works if the clock survives the resize to 640x360. Two things make it
uncertain:

1. The clock is drawn with **fixed `cv2.putText` parameters at native resolution**, and
   our videos span 640x360 to 1280x720 -- so its size *relative to the frame* varies 2x.
2. The challenge height-normalises to <=576 px before we ever see the video. Whether it
   burns the clock before or after that step changes the apparent size again.

This notebook measures the glyph height in pixels after each path and shows the crops
so the small end can be eyeballed.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

ROOT = "/projects/datasets_ML/orena"
TARGET = (640, 360)      # what training feeds the model
MAX_H = 576              # the challenge's height cap
PROBE_FRAME = "frame0001000.jpg"

# The clock sits in the black letterbox at the documented draw origin (20, 50).
# Box it tightly so bright tissue elsewhere in the top-left is not measured as text.
CLOCK_BOX = (10, 62, 15, 245)   # row0, row1, col0, col1 at NATIVE resolution

## 1. One video per native resolution

In [ ]:
def find_one_per_resolution():
    found = {}
    for ds in ("heico", "lapchole"):
        for d in sorted(glob.glob(f"{ROOT}/{ds}/frames_overlay/*/")):
            path = d + PROBE_FRAME
            try:
                size = Image.open(path).size
            except Exception:
                continue
            found.setdefault(size, (ds, d.rstrip('/').split('/')[-1], path))
    return dict(sorted(found.items(), key=lambda kv: -kv[0][1]))


videos = find_one_per_resolution()
for size, (ds, stem, _) in videos.items():
    print(f"{size[0]:5d}x{size[1]:<5d} {ds:9s} {stem[:45]}")

## 2. Measure the clock at native resolution

White text on the black letterbox, so a brightness threshold inside `CLOCK_BOX`
isolates it. Height is the ink extent in rows.

In [ ]:
def clock_crop(img, box=CLOCK_BOX, scale=1.0):
    r0, r1, c0, c1 = (int(round(v * scale)) for v in box)
    return np.asarray(img.convert("L"))[r0:r1, c0:c1]


def ink_extent(crop, thresh=200):
    ys, xs = np.where(crop > thresh)
    if len(ys) == 0:
        return 0, 0, 0.0
    return (ys.max() - ys.min() + 1, xs.max() - xs.min() + 1,
            100.0 * (crop > thresh).mean())


native = {}
for size, (ds, stem, path) in videos.items():
    img = Image.open(path)
    h, w, cover = ink_extent(clock_crop(img))
    native[size] = dict(ds=ds, stem=stem, path=path, glyph_h=h, glyph_w=w, ink=cover)
    print(f"{size[0]:5d}x{size[1]:<5d} glyph {h:3d}x{w:3d} px  "
          f"= {100*h/size[1]:4.1f}% of frame height   ink {cover:4.1f}%")

## 3. The two paths to 640x360

* **ours** -- native -> 640x360, what training does today.
* **challenge** -- native -> height 576 (aspect preserved) -> 640x360, what the
  submission container receives *if* the clock is burned before normalisation.

If the organizers burn it *after* normalising, the glyph stays ~36 px on a 576-tall
frame and only our final resize shrinks it -- a third case, computed below as
`burn_after`.

In [ ]:
def to_target(img):
    return img.resize(TARGET, Image.BILINEAR)


def height_normalise(img, max_h=MAX_H):
    w, h = img.size
    if h <= max_h:
        return img
    return img.resize((round(w * max_h / h), max_h), Image.BILINEAR)


rows = []
crops = {}
for size, info in native.items():
    img = Image.open(info["path"])
    w, h = size

    ours = to_target(img)
    chal = to_target(height_normalise(img))
    # scale the measurement box the same way the image was scaled
    s_ours = TARGET[1] / h
    s_chal = TARGET[1] / h
    gh_ours = ink_extent(clock_crop(ours, scale=s_ours))[0]
    gh_chal = ink_extent(clock_crop(chal, scale=s_chal))[0]
    burn_after = info["glyph_h"] * TARGET[1] / min(h, MAX_H)

    crops[size] = dict(ours=clock_crop(ours, scale=s_ours),
                       chal=clock_crop(chal, scale=s_chal))
    rows.append((f"{w}x{h}", info["ds"], info["glyph_h"], gh_ours, gh_chal, burn_after))

print(f"{'native':>10} {'set':>9} {'glyph@native':>13} {'ours':>6} {'challenge':>10} {'burn_after':>11}")
for r in rows:
    print(f"{r[0]:>10} {r[1]:>9} {r[2]:>13} {r[3]:>6} {r[4]:>10} {r[5]:>11.1f}")
print("\nall figures are glyph height in pixels once the frame is 640x360")

## 4. Look at it

Nearest-neighbour zoom so the actual pixels are visible -- a clock that measures fine
but reads as mush is still useless to the model.

In [ ]:
order = sorted(crops, key=lambda s: -s[1])
fig, axes = plt.subplots(len(order), 2, figsize=(13, 1.5 * len(order)))
for ax_row, size in zip(np.atleast_2d(axes), order):
    for ax, key, label in zip(ax_row, ("ours", "chal"), ("ours", "challenge")):
        ax.imshow(crops[size][key], cmap="gray", vmin=0, vmax=255,
                  interpolation="nearest")
        ax.set_title(f"{size[0]}x{size[1]} -> {label}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Does it survive at lower frame sizes?

If VRAM or latency pushes the frame size below 640x360, this is the curve that says
how far it can go before the clock stops being readable.

In [ ]:
SIZES = [(854, 480), (640, 360), (512, 288), (448, 252), (384, 216)]
worst = max(videos, key=lambda s: s[1])      # tallest source shrinks the clock most
img = Image.open(native[worst]["path"])

fig, axes = plt.subplots(len(SIZES), 1, figsize=(7, 1.2 * len(SIZES)))
for ax, size in zip(np.atleast_1d(axes), SIZES):
    c = clock_crop(img.resize(size, Image.BILINEAR), scale=size[1] / worst[1])
    ax.imshow(c, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(f"{size[0]}x{size[1]}  glyph {ink_extent(c)[0]} px", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"source: {worst[0]}x{worst[1]} ({native[worst]['ds']})")

## What to conclude

The decision this notebook feeds:

* **If the glyph stays above ~15 px at 640x360 across every source resolution**, the
  overlay arm is safe and the plan's `--frames-folder frames_overlay` switch can go
  ahead as the default.
* **If `ours` and `challenge` differ materially**, training must reproduce the
  challenge's height-normalisation step before resizing, or the model sees a clock at
  a size it never trained on.
* **If the small end is illegible**, either keep the text markers as the primary
  timestamp mechanism, or crop-and-paste the clock region at native resolution into
  the resized frame -- it lives in dead letterbox pixels, so it costs nothing.